# Active learning, one acquisition at a time
### A CPU-only training-school laboratory using handwritten digits

**What you will build:** a reproducible pool-based active-learning experiment with a classifier, scoring policy, acquisition budget, oracle, pool state, and validation history. The goal is not to maximize accuracy at any cost; it is to learn how to choose **which labels to request**.

**Core laboratory (~75–100 minutes):** understand splits and a tiny labeled baseline, calculate uncertainty, query a simulated oracle, run complete cycles, then compare acquisition policies. **Optional exercises (~20–40 minutes):** implement scoring, vary budgets, repeat seeds, and add diversity.

**Everything runs on a CPU, completely offline.** We use the built-in scikit-learn digits dataset and the *actual* \`al_pipeline.core\` interfaces. The science-specific astronomical adapters are not needed.

> **Important distinction:** The model must never see labels from the unlabeled candidate pool. The benchmark stores those labels *only* so a simulated oracle can reveal them after a selection. We tune policies against a fixed validation split and reserve the held-out test split for one final evaluation.


## 1. The experimental question and the four data populations

In standard supervised learning, we start with labels for all training objects. In active learning, we start with a small labeled set \(L_0\) and a larger unlabeled candidate pool \(U_0\). At cycle \(t\):

1. Train a model on **observed** labels in \(L_t\).
2. Predict class probabilities for the eligible objects in \(U_t\).
3. Rank candidates using an acquisition policy (random, entropy, margin, ...).
4. Select a batch \(B_t\), request labels from an **oracle**, and update the pool.
5. Retrain and measure validation performance at the same cumulative label budget.

We also keep a **validation** split for comparisons and a **sealed test** split for a final evaluation. Neither is ever an acquisition candidate.

The key question is: **with the same initial labels and acquisition budget, does the choice of queried objects improve classification?** A result on one seed is not conclusive.


In [ ]:
# All dependencies are CPU-friendly. Run from an editable installation:
#     python -m pip install -e ".[teaching]"
#
# RUN_EXERCISES is False so this notebook runs end-to-end *before* TODOs
# are filled in. Set it True only after completing the exercise functions.
RUN_EXERCISES = False
RUN_OPTIONAL_EXPERIMENTS = False

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from al_pipeline.core.batch_selection import DiversitySelector
from al_pipeline.core.evaluation.history import EvaluationHistory
from al_pipeline.core.evaluation.metrics import compute_classification_metrics
from al_pipeline.core.evaluation.plotting import plot_strategy_comparison
from al_pipeline.core.loop import ActiveLearningLoop
from al_pipeline.core.oracle.existing_labels import ExistingLabelsOracle
from al_pipeline.core.pool import Pool
from al_pipeline.core.scoring.entropy import EntropyScorer
from al_pipeline.core.scoring.margin import MarginScorer
from al_pipeline.core.scoring.random import RandomScorer
from al_pipeline.core.selection import TopKSelector
from al_pipeline.core.types import ClassificationPredictions

SEED = 2026
INITIAL_PER_CLASS = 5
BATCH_SIZE = 20
N_CYCLES = 8
CLASS_LABELS = tuple(range(10))
METRICS = ("accuracy", "balanced_accuracy", "f1_macro")
print("CPU-only notebook configuration:", dict(seed=SEED, initial=50,
      batch=BATCH_SIZE, cycles=N_CYCLES))


## 2. Inspect the real images (no downloads)

Scikit-learn ships 1,797 grayscale digit images. Each is only **8×8 pixels**: expressive enough for a real multiclass classification task and small enough that a linear model trains quickly.

The image is the feature input. A label such as “7” is *ground truth* that the training model may access **only after** it belongs to the labeled pool. The split is made once so every acquisition policy gets the same task.


In [ ]:
digits = load_digits()
images = np.asarray(digits.images, dtype=float)
X = np.asarray(digits.data, dtype=float) / 16.0  # pixels scaled to [0, 1]
y = np.asarray(digits.target, dtype=int)
all_ids = np.asarray([f"digit-{i:04d}" for i in range(len(y))], dtype=object)
id_to_index = {object_id: i for i, object_id in enumerate(all_ids)}
assert images.shape[1:] == (8, 8)
assert X.shape[1] == 64 and set(np.unique(y)) == set(CLASS_LABELS)

fig, axes = plt.subplots(2, 10, figsize=(13, 3.2))
for ax, i in zip(axes.flat, range(20)):
    ax.imshow(images[i], cmap="gray_r", vmin=0, vmax=16)
    ax.set_title(str(y[i]))
    ax.axis("off")
fig.suptitle("Sample handwritten digit images (labels shown here only for orientation)")
plt.tight_layout()
plt.show()


In [ ]:
# Stratified, disjoint splits: 60% candidate/training population,
# 20% validation, and 20% sealed final test.
indices = np.arange(len(y))
pool_indices, test_indices = train_test_split(
    indices, test_size=0.20, stratify=y, random_state=SEED,
)
pool_indices, val_indices = train_test_split(
    pool_indices, test_size=0.25, stratify=y[pool_indices], random_state=SEED,
)
pool_indices = np.sort(pool_indices)
val_indices = np.sort(val_indices)
test_indices = np.sort(test_indices)
assert len(set(pool_indices) & set(val_indices)) == 0
assert len(set(pool_indices) & set(test_indices)) == 0
assert len(set(val_indices) & set(test_indices)) == 0
assert len(pool_indices) + len(val_indices) + len(test_indices) == len(y)

pool_ids = all_ids[pool_indices].tolist()
print(f"Candidate/training population: {len(pool_indices)} images")
print(f"Fixed validation split: {len(val_indices)} images")
print(f"Sealed test split: {len(test_indices)} images")


### Construct \(L_0\): our first fifty labels

We select **five labeled digits per class**. For this controlled benchmark only, the dataset's existing labels are used to create that balanced initial seed. In a real scientific problem, an initial labeled catalog or a separate labeling campaign would be required.

All other candidate labels remain hidden in an \`ExistingLabelsOracle\`. The training code below uses a separate \`observed_labels\` dictionary, and the oracle is called **only for selected objects**. This separation is more meaningful than simply remembering not to inspect the labels.


In [ ]:
seed_rng = np.random.default_rng(SEED)
initial_indices = np.sort(np.concatenate([
    seed_rng.choice(pool_indices[y[pool_indices] == digit],
                    size=INITIAL_PER_CLASS, replace=False)
    for digit in CLASS_LABELS
]))
initial_ids = all_ids[initial_indices].tolist()
initial_labels = {str(all_ids[i]): int(y[i]) for i in initial_indices}

# The simulated oracle holds all candidate labels, but the model never
# receives this mapping. Only oracle.query(selection) reveals new ones.
oracle = ExistingLabelsOracle({
    str(all_ids[i]): int(y[i]) for i in pool_indices
})
baseline_pool = Pool.from_ids(pool_ids, labeled_ids=initial_ids)
assert len(initial_labels) == 50
assert len(baseline_pool.labeled_ids) == 50
assert len(baseline_pool.unlabeled_ids) == len(pool_indices) - 50
assert len(baseline_pool.pending_ids) == 0
print("Initial labeled distribution:", pd.Series(list(initial_labels.values())).value_counts().sort_index().to_dict())
print("L0:", len(baseline_pool.labeled_ids), "U0:", len(baseline_pool.unlabeled_ids))


## 3. Train and evaluate an intentionally modest model

For clarity we use **multinomial logistic regression** rather than a neural network. It predicts a full ten-class probability vector and fits quickly on a CPU. We retrain a fresh instance at each cycle, using the same settings and the cumulative observed labels.

This toy classifier is **not** an AstroMiNN architecture. The *active-learning infrastructure* is shared; the learning model is deliberately replaceable.

Our primary metrics are macro-F1 (treats classes equally), balanced accuracy (average recall across classes), and ordinary accuracy. We evaluate only on the fixed validation split while developing acquisition policies.


In [ ]:
def fit_model(observed_labels):
    """Train from scratch on exactly the labels revealed to this run."""
    ids = sorted(observed_labels)
    rows = np.asarray([id_to_index[object_id] for object_id in ids], dtype=int)
    targets = np.asarray([observed_labels[object_id] for object_id in ids], dtype=int)
    model = LogisticRegression(C=1.0, max_iter=500, solver="lbfgs")
    model.fit(X[rows], targets)
    assert np.array_equal(model.classes_, np.asarray(CLASS_LABELS))
    return model


def evaluate_validation(model):
    predictions = ClassificationPredictions(
        values=model.predict_proba(X[val_indices]),
        output_type="probabilities",
    )
    return compute_classification_metrics(
        y[val_indices], predictions, metrics=METRICS, labels=CLASS_LABELS,
    )


baseline_model = fit_model(initial_labels)
baseline_metrics = evaluate_validation(baseline_model)
display(pd.DataFrame([baseline_metrics]).round(3))


## 4. What does “uncertain” actually mean?

Suppose the model assigns probabilities \(p(c \mid x)\) to each class. **Predictive entropy** measures how dispersed that distribution is:

\[
H(x)=-\sum_{c=1}^C p(c\mid x)\log p(c\mid x).
\]

For a near-certain prediction entropy is near zero. If all ten classes receive equal probability, entropy reaches \(\log 10\). **High entropy is a ranking signal, not proof that a sample is useful.** It can also reflect low-quality or unrepresentative inputs.

By contrast, **margin acquisition** examines the gap between the largest and second-largest probabilities:

\[
m(x)=p_{(1)}(x)-p_{(2)}(x).
\]

A *smaller* gap indicates more ambiguity between the two most likely classes. Our \`MarginScorer\` returns \(-m(x)\), because the package consistently treats **larger scores as higher acquisition priority**.


In [ ]:
examples = np.asarray([
    [0.95, 0.03, 0.02],
    [0.36, 0.34, 0.30],
    [0.55, 0.44, 0.01],
], dtype=float)
safe_log = np.where(examples > 0, np.log(examples), 0.0)
manual_H = -np.sum(examples * safe_log, axis=1)
package_H = EntropyScorer().score(
    ClassificationPredictions(examples, output_type="probabilities")
)
assert np.allclose(manual_H, package_H)
display(pd.DataFrame({"probabilities": [str(row) for row in examples],
                      "entropy": package_H}).round(4))
print("Maximum three-class entropy:", round(np.log(3), 4))


In [ ]:
# Predict on the U0 candidate pool: no candidate labels are used here.
candidate_ids = sorted(baseline_pool.unlabeled_ids)
candidate_rows = np.asarray([id_to_index[oid] for oid in candidate_ids])
candidate_probabilities = baseline_model.predict_proba(X[candidate_rows])
candidate_predictions = ClassificationPredictions(
    candidate_probabilities, output_type="probabilities"
)
entropy_scores = EntropyScorer().score(candidate_predictions)
margin_scores = MarginScorer().score(candidate_predictions)
assert entropy_scores.shape == margin_scores.shape == (len(candidate_ids),)

def show_gallery(ids, title, scores=None, n=12):
    """Visualize selected objects by ID, without consulting hidden labels."""
    ids = list(ids)[:n]
    fig, axes = plt.subplots(2, 6, figsize=(12, 4))
    for ax, j in zip(axes.flat, range(12)):
        if j < len(ids):
            idx = id_to_index[ids[j]]
            ax.imshow(images[idx], cmap="gray_r", vmin=0, vmax=16)
            label = ids[j].replace("digit-", "#")
            if scores is not None:
                label += f"  score={float(scores[j]):.2f}"
            ax.set_title(label, fontsize=9)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

uncertain_order = np.argsort(-entropy_scores, kind="stable")
certain_order = np.argsort(entropy_scores, kind="stable")
show_gallery([candidate_ids[j] for j in uncertain_order[:12]],
             "Highest-entropy unlabeled images", entropy_scores[uncertain_order[:12]])
show_gallery([candidate_ids[j] for j in certain_order[:12]],
             "Lowest-entropy unlabeled images", entropy_scores[certain_order[:12]])


### TODO 1 — Reimplement predictive entropy (introductory)

Implement \`student_entropy(probabilities)\` without calling \`EntropyScorer\`. Use NumPy and support zero probabilities without computing \(\log 0\). The function should accept an \(N \times C\) array and return \(N\) scores.

**Think:** Why is comparing raw logits with entropy mathematically incorrect? What is the maximum ten-class entropy?

The following exercise cell is **deliberately disabled by default**. Fill in the function, set \`RUN_EXERCISES=True\` in the first cell, and rerun the notebook to check your result.


In [ ]:
def student_entropy(probabilities):
    """TODO 1: return an entropy score for every row (n_objects, n_classes)."""
    # TODO: implement safe -sum(p*log(p)), handling p=0 without log(0).
    return None


if RUN_EXERCISES:
    check_probabilities = np.array([[0.5, 0.5], [1.0, 0.0], [0.2, 0.8]])
    expected = EntropyScorer().score(
        ClassificationPredictions(check_probabilities, "probabilities")
    )
    np.testing.assert_allclose(student_entropy(check_probabilities), expected)
    print("Exercise 1 passed!")


### TODO 2 — Compare entropy and margin (intermediate)

Before implementing anything, inspect the first 20 object IDs favored by entropy and by margin. **Are they identical?** Try to explain any difference using a three-class counterexample where one distribution has a similar top-two gap but a very different third-class probability.

Then implement \`student_margin\`. Remember our “larger is better” ranking convention: return the **negative** top-two gap.


In [ ]:
entropy_top = [candidate_ids[i] for i in np.argsort(-entropy_scores)[:20]]
margin_top = [candidate_ids[i] for i in np.argsort(-margin_scores)[:20]]
overlap = set(entropy_top) & set(margin_top)
print(f"Entropy/margin overlap in top 20: {len(overlap)}/20")
display(pd.DataFrame({"entropy_top": entropy_top, "margin_top": margin_top}))

def student_margin(probabilities):
    """TODO 2: negative gap between largest and second-largest class probability."""
    # TODO: implement the margin scorer in NumPy.
    return None


if RUN_EXERCISES:
    np.testing.assert_allclose(
        student_margin(candidate_probabilities[:50]),
        margin_scores[:50],
    )
    print("Exercise 2 passed!")


## 5. Ranking, selection, and pool transitions

An acquisition **scorer** produces one number per candidate. A **selector** enforces the budget and eligibility rules. Calling \`ActiveLearningLoop.query\` merely proposes a batch and does **not** change the pool. Calling \`acquire\` commits the choice by moving each object from **unlabeled → pending**.

An \`ExistingLabelsOracle\` is ideal for an offline teaching experiment: it keeps known labels inaccessible to the model until explicitly queried. Upon successful return, \`Pool.resolve\` moves the selected objects **pending → labeled**. Real oracles might respond slowly, fail, or return labels outside the taxonomy; those are important extensions, not assumptions silently baked into this toy case.

The short demonstration below uses its **own disposable pool**, so it cannot contaminate the later fair policy comparison.


In [ ]:
demo_pool = Pool.from_ids(pool_ids, labeled_ids=initial_ids)
demo_loop = ActiveLearningLoop(
    scorer=EntropyScorer(), selector=TopKSelector(), pool=demo_pool
)
proposal = demo_loop.query(
    ids=candidate_ids, acquisition_data=candidate_predictions, k=8
)
assert len(demo_pool.pending_ids) == 0  # querying is read-only
committed = demo_loop.acquire(
    ids=candidate_ids, acquisition_data=candidate_predictions, k=8
)
assert committed.ids == proposal.ids
assert set(committed.ids) == set(demo_pool.pending_ids)
print("After acquire: labeled =", len(demo_pool.labeled_ids),
      "pending =", len(demo_pool.pending_ids))
returned_labels = oracle.query(committed)
assert returned_labels.ids == committed.ids
demo_pool.resolve(returned_labels.ids)
assert len(demo_pool.pending_ids) == 0
assert len(demo_pool.labeled_ids) == 50 + 8
print("After oracle + resolve: labeled =", len(demo_pool.labeled_ids),
      "pending =", len(demo_pool.pending_ids))
show_gallery(committed.ids, "Eight objects requested from the oracle",
             scores=committed.scores)


## 6. Build the full closed loop

We can now assemble the experiment. A policy has its **own pool, observed labels, RNG and evaluation history**. Every policy starts with the identical \(L_0\), train/validation split and classifier definition.

At each cycle we train a **new** logistic-regression instance on the cumulative observed labels. This is a restart-from-scratch comparison; it avoids introducing warm-start/checkpoint complexities into the first teaching exercise.

For random acquisition, \`RandomScorer\` receives a candidate count rather than model probabilities. Entropy and margin receive \`ClassificationPredictions\`. The optional diversity policy ranks by entropy first, then uses \`DiversitySelector\` to spread a batch across a shortlist in feature space.

**Audit invariant:** the training label dictionary equals the pool's labeled IDs. We never use test labels, and unlabeled ground truth enters the run only through \`oracle.query(selection)\`.
